# Day 6: เตรียมตารางให้ Power BI

Power BI ไม่ได้รันโมเดลเอง notebook นี้จึงรวมผลจาก Day 2 ถึง Day 5 แล้วเซฟเป็น CSV 6 ไฟล์ใน `data/export/`

| ไฟล์ | 1 แถว = | ใช้ทำอะไรใน Power BI |
|---|---|---|
| `stations.csv` | 1 สถานี | ตารางมิติ เชื่อมด้วย `station_id` |
| `hourly.csv` | 1 สถานี 1 ชั่วโมง | ตารางข้อเท็จจริง กราฟเส้นและการ์ดตัวเลข |
| `episodes.csv` | 1 เหตุการณ์ | ตารางข้อเท็จจริง หน้าเหตุการณ์และ drill-through |
| `forecast.csv` | 1 ชั่วโมงที่ทาย 1 สาร | ตารางข้อเท็จจริง หน้าพยากรณ์ |
| `plants.csv` | 1 พื้นที่อุตสาหกรรม | หน้าแผนที่ |
| `wind_sectors.csv` | 1 สถานี 1 สาร 1 ทิศ | หน้าทิศลม (radar chart) |

ข้อตกลงที่ใช้ทุกไฟล์
- เวลาเป็นเวลาไทย และตัดป้าย timezone ออก เพราะ Power BI อ่าน timezone ได้ไม่ดี
- มีคอลัมน์วันที่ไว้เชื่อมกับตาราง `date` ที่จะสร้างด้วย DAX
- flag และ alert เป็น 1/0 เพื่อให้ Power BI ใช้ SUM นับได้
- เซฟเป็น `utf-8-sig` เพื่อให้ภาษาไทยไม่เพี้ยน

ข้อมูลทั้งหมดเป็นข้อมูลย้อนหลัง (ล่าสุด 2 ต.ค. 2026) ไม่ใช่ real-time แดชบอร์ดจึงเป็นการดูย้อนหลังและทดสอบย้อนหลัง

## 6.1 โหลดข้อมูล

- `hourly_merged` มีครบ 4 สถานี (จาก Day 2)
- `hourly_with_increment` มีเฉพาะสถานีใกล้นิคม พร้อม local increment และ flag เหตุการณ์ (จาก Day 3)

In [1]:
import pandas as pd

EXPORT = '../data/export/'

# merged = ทุกสถานี, with_increment = เฉพาะสถานีใกล้นิคม + คอลัมน์ increment
merged = pd.read_parquet('../data/processed/hourly_merged.parquet')
hourly_inc = pd.read_parquet('../data/processed/hourly_with_increment.parquet')
print(merged.shape, hourly_inc.shape)

(151093, 29) (66852, 69)


## 6.2 `stations.csv`

ชื่อและ role ดึงจากข้อมูลจริง ส่วนพิกัดเป็นชุดเดียวกับที่ใช้ดึงลมใน `src/fetch_wind.py` และใน notebook 04
เพิ่มป้ายภาษาไทยไว้โชว์ในแดชบอร์ด

In [2]:
# พิกัดสถานี (ชุดเดียวกับ src/fetch_wind.py และ notebook 04)
coords = pd.DataFrame({
    'station_id': [225611, 7846, 717, 225690],
    'lat': [12.7087, 12.7352, 12.9738, 12.67155],
    'lon': [101.1660, 101.1356, 101.2129, 101.2758667],
})

# ชื่อและ role เอามาจากข้อมูลจริง จะได้ตรงกับ hourly.csv
stations = merged[['station_id', 'station_name', 'role']].drop_duplicates()
stations = stations.merge(coords, on='station_id')

# ป้ายภาษาไทยไว้โชว์ใน Power BI
stations['role_th'] = stations['role'].map({
    'near': 'ใกล้นิคม',
    'background': 'อ้างอิง',
    'background_alt': 'อ้างอิงสำรอง',
})

display(stations)
stations.to_csv(EXPORT + 'stations.csv', index=False, encoding='utf-8-sig')


,station_id,station_name,role,lat,lon,role_th
0,717,Pluakdaeng District Health Office,background,12.97380,101.212900,อ้างอิง
1,7846,"Field Crop Research Center, Rayong",near,12.73520,101.135600,ใกล้นิคม
2,225611,Health Promotion Hospital Maptaput,near,12.70870,101.166000,ใกล้นิคม
3,225690,Agricultural Office,background_alt,12.67155,101.275867,อ้างอิงสำรอง


## 6.3 `hourly.csv`

- ลมใช้ IFS ก่อน ถ้าไม่มีค่อยใช้ ERA5 แบบเดียวกับ Day 4
- local increment และ flag เหตุการณ์มีเฉพาะสถานีใกล้นิคม สถานีอ้างอิงจึงเป็นช่องว่าง
- คำนวณ increment ใน Python เลย เพราะต้องจับคู่สองสถานี ทำใน DAX ยากกว่า

In [3]:
# ตารางหลัก = merged (ครบ 4 สถานี)
cols = ['station_id', 'datetime_utc', 'datetime_local',
        'co', 'no2', 'o3', 'pm10', 'pm25', 'so2']
hourly = merged[cols].copy()

# ลม: ใช้ IFS ก่อน ถ้าไม่มีค่อยใช้ ERA5 (แบบเดียวกับ Day 4)
hourly['wind_dir'] = merged['ifs_wind_direction_10m'].fillna(merged['era5_wind_direction_10m'])
hourly['wind_speed'] = merged['ifs_wind_speed_10m'].fillna(merged['era5_wind_speed_10m'])

# increment และ flag เหตุการณ์ มีเฉพาะสถานีใกล้นิคม
extra = hourly_inc[['station_id', 'datetime_utc',
                    'pm25_increment', 'so2_increment',
                    'pm25_episode_flag', 'so2_episode_flag']]
hourly = hourly.merge(extra, on=['station_id', 'datetime_utc'], how='left')

# จำนวนแถวต้องเท่าเดิม ถ้าไม่เท่าแปลว่า merge แล้วมีแถวซ้ำ
assert len(hourly) == len(merged)

# ตัด +07:00 ออก เพราะ Power BI อ่าน datetime ที่มี timezone ได้ไม่ดี
# (ค่าเวลายังเป็นเวลาไทยเหมือนเดิม แค่ไม่มีป้าย timezone)
hourly['datetime_local'] = hourly['datetime_local'].dt.tz_localize(None)

# คอลัมน์วันที่ไว้เชื่อมกับตาราง date ใน Power BI
hourly['date'] = hourly['datetime_local'].dt.date

# Power BI ใช้เวลาไทยอย่างเดียว ไม่ต้องส่ง UTC ไป
hourly = hourly.drop(columns='datetime_utc')

# flag เป็น 1/0 ให้ Power BI SUM ได้ (สถานีอ้างอิงไม่มี flag จึงเป็นช่องว่าง)
for c in ['pm25_episode_flag', 'so2_episode_flag']:
    hourly[c] = hourly[c].map({True: 1, False: 0}).astype('Int64')

display(hourly.head())
print(f'{len(hourly):,} แถว, {hourly.shape[1]} คอลัมน์')
print(hourly['pm25_episode_flag'].value_counts(dropna=False))
print(hourly['so2_episode_flag'].value_counts(dropna=False))
hourly.to_csv(EXPORT + 'hourly.csv', index=False, encoding='utf-8-sig')


,station_id,datetime_local,co,no2,o3,pm10,pm25,so2,wind_dir,wind_speed,pm25_increment,so2_increment,pm25_episode_flag,so2_episode_flag,date
0,717,2016-01-30 07:00:00,NaN,0.013,0.037,113.0,NaN,0.001,107.0,3.8,NaN,NaN,<NA>,<NA>,2016-01-30
1,717,2016-01-30 08:00:00,NaN,0.009,0.038,110.0,NaN,0.002,111.0,3.1,NaN,NaN,<NA>,<NA>,2016-01-30
2,717,2016-01-30 09:00:00,NaN,0.016,0.031,107.0,NaN,0.006,169.0,3.7,NaN,NaN,<NA>,<NA>,2016-01-30
3,717,2016-01-30 10:00:00,NaN,0.005,0.048,104.0,NaN,0.002,188.0,5.1,NaN,NaN,<NA>,<NA>,2016-01-30
4,717,2016-01-30 11:00:00,NaN,0.003,0.053,102.0,NaN,0.001,188.0,5.1,NaN,NaN,<NA>,<NA>,2016-01-30


151,093 แถว, 15 คอลัมน์
pm25_episode_flag
<NA>    84241
0       63821
1        3031
Name: count, dtype: Int64
so2_episode_flag
<NA>    84241
0       63784
1        3068
Name: count, dtype: Int64


**อ่านผล:** 151,093 แถว 15 คอลัมน์ เท่ากับ `hourly_merged` จึงไม่มีแถวซ้ำจากการ merge
flag ที่เป็นช่องว่าง 84,241 แถว คือแถวของสถานีอ้างอิง (151,093 − 66,852)

## 6.4 `episodes.csv`

ใช้ป้ายจาก Day 4 (กฎ `regional` เป็นฉบับที่แก้ในหัวข้อ 4.15)
เหตุผลของแต่ละป้ายอยู่ใน 3 คอลัมน์
- `frac_wind_from_industry`: สัดส่วนชั่วโมงที่ลมพัดมาจากทิศนิคม
- `ctrl_ratio`: สถานีอ้างอิงสูงตามด้วยแค่ไหน
- `median_increment` เทียบกับ `inc_threshold`: สถานีใกล้นิคมสูงกว่าสถานีอ้างอิงเกินเกณฑ์หรือไม่

เชื่อมกับตาราง `date` ด้วยวันที่เริ่มเหตุการณ์ (`start_date`) ตามแผน

In [4]:
ep = pd.read_csv('../data/processed/episodes_labeled.csv')

# id ไว้ใช้ทำ drill-through ใน Power BI
ep['episode_id'] = range(1, len(ep) + 1)

# แปลงเวลา UTC เป็นเวลาไทย แล้วตัดป้าย timezone ออกเหมือน hourly.csv
ep['start_local'] = pd.to_datetime(ep['start_utc']).dt.tz_convert('Asia/Bangkok').dt.tz_localize(None)
ep['end_local'] = pd.to_datetime(ep['end_utc']).dt.tz_convert('Asia/Bangkok').dt.tz_localize(None)

# เชื่อมกับตาราง date ด้วยวันที่เริ่มเหตุการณ์ (ตามแผน)
ep['start_date'] = ep['start_local'].dt.date

# ป้ายภาษาไทยไว้โชว์ใน Power BI
ep['label_th'] = ep['label'].map({
    'local-consistent': 'สอดคล้องกับแหล่งในพื้นที่',
    'regional': 'สอดคล้องกับมลพิษระดับภูมิภาค',
    'unclear-mixed': 'ไม่ชัดเจน (สัญญาณปนกัน)',
    'unclear-no-data': 'ไม่ชัดเจน (ข้อมูลไม่พอ)',
})

# เก็บเฉพาะคอลัมน์ที่ใช้ใน dashboard
# ส่วน "เหตุผล" ของป้าย = สัดส่วนลมจากนิคม, ctrl_ratio, increment เทียบ threshold
keep = ['episode_id', 'station_id', 'pollutant',
        'start_local', 'end_local', 'start_date', 'duration_hours',
        'peak_value', 'season', 'label', 'label_th',
        'frac_wind_from_industry', 'ctrl_ratio', 'median_increment', 'inc_threshold']
episodes = ep[keep]

# ทุกป้ายต้องแปลไทยได้ ถ้ามีค่าว่างแปลว่ามีป้ายที่ไม่ได้ใส่ใน map
assert episodes['label_th'].notna().all()

display(episodes.head())
display(episodes['label_th'].value_counts().to_frame())
episodes.to_csv(EXPORT + 'episodes.csv', index=False, encoding='utf-8-sig')


,episode_id,station_id,pollutant,start_local,end_local,start_date,duration_hours,peak_value,season,label,label_th,frac_wind_from_industry,ctrl_ratio,median_increment,inc_threshold
0,1,7846,so2,2016-04-08 15:00:00,2016-04-08 15:00:00,2016-04-08,1,0.011,summer,local-consistent,สอดคล้องกับแหล่งในพื้นที่,1.0,0.181818,0.009,0.002
1,2,7846,so2,2016-04-10 15:00:00,2016-04-10 17:00:00,2016-04-10,3,0.018,summer,local-consistent,สอดคล้องกับแหล่งในพื้นที่,1.0,0.161290,0.013,0.002
2,3,7846,so2,2016-04-11 15:00:00,2016-04-11 16:00:00,2016-04-11,2,0.027,summer,local-consistent,สอดคล้องกับแหล่งในพื้นที่,1.0,0.095238,0.019,0.002
3,4,7846,so2,2016-04-11 23:00:00,2016-04-11 23:00:00,2016-04-11,1,0.009,summer,local-consistent,สอดคล้องกับแหล่งในพื้นที่,1.0,0.111111,0.008,0.002
4,5,7846,so2,2016-04-12 15:00:00,2016-04-12 15:00:00,2016-04-12,1,0.009,summer,local-consistent,สอดคล้องกับแหล่งในพื้นที่,1.0,0.444444,0.005,0.002


,count
label_th,
สอดคล้องกับแหล่งในพื้นที่,967
ไม่ชัดเจน (ข้อมูลไม่พอ),393
ไม่ชัดเจน (สัญญาณปนกัน),341
สอดคล้องกับมลพิษระดับภูมิภาค,95


**อ่านผล:** มีทั้งหมด 1,796 เหตุการณ์ ป้ายที่มากที่สุดคือ "สอดคล้องกับแหล่งในพื้นที่" 967 เหตุการณ์
ป้ายเป็นเพียงการจัดกลุ่มตามความสอดคล้องของลมและสถานีอ้างอิง ไม่ใช่การพิสูจน์ว่าโรงงานใดเป็นต้นเหตุ

## 6.5 `plants.csv`

ใช้ข้อมูลจาก OpenStreetMap อย่างเดียว ไม่ได้ใช้ WRI Global Power Plant Database ตามที่แผนเขียนไว้
OSM เก็บพื้นที่เป็นกรอบสี่เหลี่ยม จึงใช้จุดกึ่งกลางกรอบเป็นตำแหน่งปักหมุด

In [5]:
import json

# พื้นที่อุตสาหกรรมและโรงไฟฟ้าจาก OpenStreetMap (ดึงไว้ใน src/fetch_osm.py)
osm = json.load(open('../data/raw/osm_industrial_sites.json', encoding='utf-8'))

rows = []
for el in osm['elements']:
    tags = el.get('tags', {})
    b = el['bounds']
    rows.append({
        'osm_id': el['id'],
        'name': tags.get('name:en') or tags.get('name'),
        'power': tags.get('power'),            # 'plant' = โรงไฟฟ้า
        'industrial': tags.get('industrial'),  # เช่น factory, oil, refinery
        # จุดกึ่งกลางกรอบสี่เหลี่ยม ใช้ปักหมุดบนแผนที่
        'lat': (b['minlat'] + b['maxlat']) / 2,
        'lon': (b['minlon'] + b['maxlon']) / 2,
    })
plants = pd.DataFrame(rows)

# ประเภทไว้แยกสีบนแผนที่
plants['type'] = 'พื้นที่อุตสาหกรรม'
plants.loc[plants['industrial'].notna(), 'type'] = 'โรงงาน'
plants.loc[plants['power'] == 'plant', 'type'] = 'โรงไฟฟ้า'

plants = plants.drop(columns=['power', 'industrial'])

display(plants.head())
display(plants['type'].value_counts().to_frame())
print(f"{len(plants)} แห่ง, มีชื่อ {plants['name'].notna().sum()} แห่ง")
plants.to_csv(EXPORT + 'plants.csv', index=False, encoding='utf-8-sig')


,osm_id,name,lat,lon,type
0,107789748,โรงไฟฟ้าระยอง,12.760883,101.154605,โรงไฟฟ้า
1,153741069,Pinthong Industrial Estate 1,13.114461,100.998909,พื้นที่อุตสาหกรรม
2,164044500,Pinthong Industrial Estate 2,13.112503,101.032916,พื้นที่อุตสาหกรรม
3,165813605,Yokohama Tire Test Centre,13.065468,101.225638,พื้นที่อุตสาหกรรม
4,165816909,Eastern Seaboard Industrial Estate Rayong,13.014331,101.174000,พื้นที่อุตสาหกรรม


,count
type,
พื้นที่อุตสาหกรรม,203
โรงไฟฟ้า,40
โรงงาน,34


277 แห่ง, มีชื่อ 72 แห่ง


**อ่านผล:** 277 แห่ง มีชื่อแค่ 72 แห่ง ส่วนใหญ่เป็นพื้นที่อุตสาหกรรม (203) รองลงมาเป็นโรงไฟฟ้า (40) และโรงงาน (34)

## 6.6 `wind_sectors.csv`

Power BI ไม่มีกราฟโพลาร์ จึงคำนวณ pollution rose และ CPF ไว้ก่อน แบ่งทิศลมเป็น 16 ช่วง ช่วงละ 22.5 องศา
CPF คือความน่าจะเป็นที่ค่าจะอยู่ใน 25% สูงสุด เมื่อลมมาจากทิศนั้น (วิธีเดียวกับ notebook 04 หัวข้อ 4.12)
ทิศของนิคมใช้ผลจาก notebook 04 หัวข้อ 4.4

In [6]:
# CPF = ความน่าจะเป็นที่ค่าจะอยู่ใน 25% สูงสุด เมื่อลมมาจากทิศนั้น (วิธีเดียวกับ notebook 04 หัวข้อ 4.12)
LABELS = ['N', 'NNE', 'NE', 'ENE', 'E', 'ESE', 'SE', 'SSE',
          'S', 'SSW', 'SW', 'WSW', 'W', 'WNW', 'NW', 'NNW']

# ทิศที่นิคมตั้งอยู่ (ผลจาก notebook 04 หัวข้อ 4.4)
INDUSTRY_DIRS = {
    225611: ['SSW', 'SW', 'WSW', 'W', 'WNW'],
    7846: ['NE', 'ENE', 'SE', 'SSE', 'S', 'SSW', 'SW'],
}

# แบ่งทิศลมเป็น 16 ช่วง ช่วงละ 22.5 องศา (0 = N, 1 = NNE, ...)
near = hourly[hourly['station_id'].isin(INDUSTRY_DIRS)].copy()
near['sector'] = ((near['wind_dir'] + 11.25) % 360) // 22.5

tables = []
for sid in INDUSTRY_DIRS:
    for p in ['pm25', 'so2']:
        g = near[(near['station_id'] == sid) & near[p].notna() & near['sector'].notna()].copy()
        top_q = g[p].quantile(0.75)
        g['high'] = g[p] >= top_q

        t = g.groupby('sector').agg(
            n_hours=(p, 'size'),
            mean_value=(p, 'mean'),
            cpf=('high', 'mean'),
        ).reset_index()
        t['station_id'] = sid
        t['pollutant'] = p
        t['top25_threshold'] = top_q
        tables.append(t)

wind_sectors = pd.concat(tables, ignore_index=True)

# ชื่อทิศ, องศากลางช่วง (ไว้วาด radar chart) และบอกว่าเป็นทิศนิคมไหม
wind_sectors['sector'] = wind_sectors['sector'].astype(int)
wind_sectors['direction'] = wind_sectors['sector'].map(lambda i: LABELS[i])
wind_sectors['degree'] = wind_sectors['sector'] * 22.5
wind_sectors['is_industry_dir'] = [d in INDUSTRY_DIRS[s] for s, d in
                                   zip(wind_sectors['station_id'], wind_sectors['direction'])]

# ต้องได้ 2 สถานี x 2 สาร x 16 ทิศ = 64 แถว
assert len(wind_sectors) == 64

wind_sectors = wind_sectors[['station_id', 'pollutant', 'sector', 'direction', 'degree',
                             'is_industry_dir', 'n_hours', 'mean_value', 'cpf', 'top25_threshold']]
display(wind_sectors.head(16))

# เช็กกับ notebook 04: CPF เฉลี่ยทิศนิคม เทียบทิศอื่น
check = wind_sectors.groupby(['station_id', 'pollutant', 'is_industry_dir'])['cpf'].mean().round(3).unstack()
display(check)

wind_sectors.to_csv(EXPORT + 'wind_sectors.csv', index=False, encoding='utf-8-sig')


,station_id,pollutant,sector,direction,degree,is_industry_dir,n_hours,mean_value,cpf,top25_threshold
0,225611,pm25,0,N,0.0,False,1101,26.882016,0.429609,26.1
1,225611,pm25,1,NNE,22.5,False,3190,25.531348,0.413166,26.1
2,225611,pm25,2,NE,45.0,False,1891,24.594183,0.368059,26.1
3,225611,pm25,3,ENE,67.5,False,546,24.611172,0.351648,26.1
4,225611,pm25,4,E,90.0,False,431,24.415313,0.354988,26.1
5,225611,pm25,5,ESE,112.5,False,431,24.369142,0.352668,26.1
6,225611,pm25,6,SE,135.0,False,654,24.536697,0.357798,26.1
7,225611,pm25,7,SSE,157.5,False,1529,23.462132,0.304120,26.1
8,225611,pm25,8,S,180.0,False,3821,23.114551,0.275059,26.1
9,225611,pm25,9,SSW,202.5,True,4230,21.377494,0.246572,26.1


is_industry_dir       False  True 
station_id pollutant              
7846       pm25       0.336  0.286
           so2        0.225  0.332
225611     pm25       0.359  0.171
           so2        0.121  0.340

**อ่านผล:** ตารางล่างเทียบ CPF เฉลี่ยของทิศนิคม (`True`) กับทิศอื่น (`False`)
- SO2: ทิศนิคมสูงกว่าทั้งสองสถานี (7846: 0.332 กับ 0.225, 225611: 0.340 กับ 0.121) สอดคล้องกับแหล่งในพื้นที่
- PM2.5: ทิศนิคมต่ำกว่า (7846: 0.286 กับ 0.336, 225611: 0.171 กับ 0.359) PM2.5 ที่สูงจึงไม่ได้มากับลมจากนิคมเป็นหลัก

## 6.7 `forecast.csv`

ค่าจริงเทียบค่าพยากรณ์ล่วงหน้า 24 ชม. ของชุด test (จาก notebook 05 หัวข้อ 5.8) พร้อมคอลัมน์ baseline
เพิ่มคอลัมน์ "เตือน" เพื่อทดสอบย้อนหลังว่าถ้าใช้เตือนจริงในปี 2025 จะเตือนถูกแค่ไหน ไม่ใช่ระบบเตือนที่ใช้งานจริง

เกณฑ์ที่ใช้
- PM2.5 37.5 µg/m³ เป็นมาตรฐานเฉลี่ย 24 ชม. แต่ที่นี่นำมาเทียบรายชั่วโมง จึงเรียกว่า "เกณฑ์อ้างอิง" ไม่ใช่ "เกินมาตรฐาน"
- SO2 0.1 ppm (100 ppb) เป็นมาตรฐานเฉลี่ย 1 ชม. ฉบับปี 2569 (ยืนยันจากแหล่งรอง ยังไม่ได้ตรวจกับราชกิจจานุเบกษาฉบับจริง)

ค่าถึงเกณฑ์ (`>=`) นับว่าเตือน และ `date` คือวันที่ของชั่วโมงที่ถูกทาย (`target_time`)

In [7]:
# ค่าจริง vs ค่าพยากรณ์ของชุด test (ได้มาจาก notebook 05 หัวข้อ 5.8)
fc = pd.read_csv('../data/processed/forecast_test.csv',
                 parse_dates=['issued_at', 'target_time'])

# เกณฑ์อ้างอิง (PM2.5 เป็นเกณฑ์เฉลี่ย 24 ชม. แต่เรานำมาเทียบรายชั่วโมง จึงเรียกว่า "เกณฑ์อ้างอิง")
THRESHOLDS = {'pm25': 37.5,    # ug/m3
              'so2': 0.1}      # ppm (= 100 ppb เฉลี่ย 1 ชม.)
fc['threshold'] = fc['pollutant'].map(THRESHOLDS)

# alert = 1 เมื่อค่าถึงเกณฑ์ (ใช้ 1/0 เหมือน flag ใน hourly.csv)
fc['actual_alert'] = (fc['target'] >= fc['threshold']).astype(int)
fc['persistence_alert'] = (fc['persistence'] >= fc['threshold']).astype(int)
fc['lightgbm_alert'] = (fc['lightgbm'] >= fc['threshold']).astype(int)

# วันที่ของชั่วโมงที่ถูกทาย ไว้เชื่อมกับตาราง date ใน Power BI
fc['date'] = fc['target_time'].dt.date

# แถวต้องเท่ากับ 5.8 (PM2.5 5,756 + SO2 1,903)
assert len(fc) == 5756 + 1903

display(fc.head())
fc.to_csv(EXPORT + 'forecast.csv', index=False, encoding='utf-8-sig')


# precision = เตือนแล้วเกิดจริงกี่ %, recall = ของที่เกิดจริงเตือนทันกี่ %
# ถ้าตัวหารเป็น 0 (เช่น ไม่ได้เตือนเลย) ใส่ NaN = คำนวณไม่ได้ ไม่ใช่ 0
def alert_scores(g, method):
    pred, real = g[method + '_alert'], g['actual_alert']
    tp = ((pred == 1) & (real == 1)).sum()
    fp = ((pred == 1) & (real == 0)).sum()
    fn = ((pred == 0) & (real == 1)).sum()
    return pd.Series({
        'real_exceed_hours': real.sum(),
        'alerts': pred.sum(),
        'hit': tp,
        'precision': tp / (tp + fp) if tp + fp > 0 else float('nan'),
        'recall': tp / (tp + fn) if tp + fn > 0 else float('nan'),
    })


rows = []
for p, g in fc.groupby('pollutant'):
    for m in ['persistence', 'lightgbm']:
        rows.append(alert_scores(g, m).rename((p, m)))
scores = pd.DataFrame(rows)
scores.index.names = ['pollutant', 'method']
display(scores.round(3))

,station_id,pollutant,issued_at,target_time,target,persistence,naive_week,lightgbm,threshold,actual_alert,persistence_alert,lightgbm_alert,date
0,225611,pm25,2024-12-31 08:00:00,2025-01-01 08:00:00,37.4,37.3,27.0,34.342546,37.5,0,0,0,2025-01-01
1,225611,pm25,2024-12-31 09:00:00,2025-01-01 09:00:00,37.7,37.5,26.8,33.858909,37.5,1,1,0,2025-01-01
2,225611,pm25,2024-12-31 10:00:00,2025-01-01 10:00:00,38.9,36.6,26.5,36.116834,37.5,1,0,0,2025-01-01
3,225611,pm25,2024-12-31 11:00:00,2025-01-01 11:00:00,39.6,35.9,26.0,35.001298,37.5,1,0,0,2025-01-01
4,225611,pm25,2024-12-31 13:00:00,2025-01-01 13:00:00,40.0,36.2,26.8,34.528024,37.5,1,0,0,2025-01-01


real_exceed_hours  alerts    hit  precision  recall
pollutant method                                                          
pm25      persistence              673.0   622.0  427.0      0.686   0.634
          lightgbm                 673.0   259.0  208.0      0.803   0.309
so2       persistence                6.0     2.0    0.0      0.000   0.000
          lightgbm                   6.0     0.0    0.0        NaN   0.000

**อ่านผล**
- PM2.5: LightGBM เตือนน้อยแต่แม่นกว่า (precision 0.803, recall 0.309) ส่วน `persistence` เตือนบ่อยกว่าและจับได้มากกว่า (precision 0.686, recall 0.634)
- SO2: ในชุด test เกินเกณฑ์จริงแค่ 6 ชั่วโมง จึงน้อยเกินกว่าจะสรุปได้ `persistence` เตือน 2 ครั้งและผิดทั้งคู่ ส่วน LightGBM ไม่เตือนเลย precision จึงเป็น `NaN` (คำนวณไม่ได้ เพราะหารด้วย 0)

LightGBM เตือนน้อยเพราะทายค่ากระจุกอยู่ใกล้ค่ากลาง (ดูสาเหตุใน notebook 05 หัวข้อ 5.7)

## สรุป Day 6

ได้ CSV ครบ 6 ไฟล์ใน `data/export/` พร้อมใช้ใน Power BI

**สิ่งที่ต้องเขียนใน README**
1. ข้อมูลเป็นข้อมูลย้อนหลัง แดชบอร์ดจึงเป็นการดูย้อนหลังและทดสอบย้อนหลัง ไม่ใช่ real-time
2. ตำแหน่งโรงงานใช้ OSM อย่างเดียว ไม่ได้ใช้ WRI
3. PM2.5 ใช้เกณฑ์ 24 ชม. เทียบรายชั่วโมง จึงเป็น "เกณฑ์อ้างอิง"
4. เกณฑ์ SO2 ยังไม่ได้ตรวจกับเอกสารทางการฉบับจริง
5. SO2 ในชุด test เกินเกณฑ์แค่ 6 ชั่วโมง ผลการเตือน SO2 จึงสรุปไม่ได้